In [1]:
# Import necessary lib's

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Loading file

In [8]:
FILEPATH = "../data/raw/data001.json"

# reading only chunk of data, for understanding pattern
df = next(
    pd.read_json(
        FILEPATH,
        lines=True,
        chunksize=100_000
    )
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              100000 non-null  float64
 1   submitter       100000 non-null  object 
 2   authors         100000 non-null  object 
 3   title           100000 non-null  object 
 4   comments        87344 non-null   object 
 5   journal-ref     50781 non-null   object 
 6   doi             61407 non-null   object 
 7   report-no       9466 non-null    object 
 8   categories      100000 non-null  object 
 9   license         56782 non-null   object 
 10  abstract        100000 non-null  object 
 11  versions        100000 non-null  object 
 12  update_date     100000 non-null  object 
 13  authors_parsed  100000 non-null  object 
dtypes: float64(1), object(13)
memory usage: 10.7+ MB


In [9]:
df.shape

(100000, 14)

In [32]:
df.sample(1)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
90055,810.4476,John Thrower,"J. D. Thrower, M. P. Collings, F. J. M. Rutten...",Laboratory investigations of the interaction b...,"23 pages, including 6 figures and 1 table ; Su...","MNRAS, 394, 1510 (2009)",10.1111/j.1365-2966.2009.14420.x,None,astro-ph cond-mat.mtrl-sci,http://arxiv.org/licenses/nonexclusive-distrib...,Experimental results on the thermal desorpti...,"[{'version': 'v1', 'created': 'Fri, 24 Oct 200...",2015-10-21,"[[Thrower, J. D., ], [Collings, M. P., ], [Rut..."


**filter out for the fields like, these are the only fields where Computer Science paper lies.**
1. cs.LG (Machine learning)
2. stat.ML (Statistical machine learning)
3. cs.CL (Computation and language {NLP})
4. cs.AI (Artifical Intelligence)
5. cs.CV (Computer vision)
6. cs.RO (Robotics)
7. cs.IR (Information retrieval)
8. cs.NE (Neural and Evolutionary computing)

In [26]:
import re

categories = [
    'cs.LG', 'stat.ML', 'cs.CL', 'cs.AI', 'cs.IR', 'cs.NE'
]

pattern = "|".join(
    map(re.escape, categories)
)

print(pattern)

cs\.LG|stat\.ML|cs\.CL|cs\.AI|cs\.IR|cs\.NE


In [62]:
for item in map(re.escape, categories):
    print(item)

cs\.LG
stat\.ML
cs\.CL
cs\.AI
cs\.IR
cs\.NE


In [52]:
cs_rows = df[
    df['categories'].str.contains(
        pattern,
        regex=True,
        na=False
    )
]

In [53]:
cs_rows.shape

(1018, 14)

In [54]:
cs_rows.sample(1)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
79241,808.3109,Florentin Smarandache,"Florentin Smarandache, V. Christianto",n-ary Fuzzy Logic and Neutrosophic Logic Opera...,"15 pages, 2 fuzzy and neutrosophic value table...","Studies in Logic, Grammar and Rethoric [Belaru...",None,None,cs.AI,http://arxiv.org/licenses/nonexclusive-distrib...,We extend Knuth's 16 Boolean binary logic op...,"[{'version': 'v1', 'created': 'Fri, 22 Aug 200...",2010-02-16,"[[Smarandache, Florentin, ], [Christianto, V., ]]"


In [55]:
FILEDS_TO_KEEP = [
    'id',
    'title',
    'abstract',
    'categories',
    'update_date',
    'authors_parsed',
    'license'
]

In [56]:
cs_rows[ FILEDS_TO_KEEP ]

,id,title,abstract,categories,update_date,authors_parsed,license
46,704.0047,Intelligent location of simultaneously active ...,The intelligent acoustic emission locator is...,cs.NE cs.AI,2009-09-29,"[[Kosel, T., ], [Grabec, I., ]]",None
49,704.0050,Intelligent location of simultaneously active ...,Part I describes an intelligent acoustic emi...,cs.NE cs.AI,2007-05-23,"[[Kosel, T., ], [Grabec, I., ]]",None
303,704.0304,The World as Evolving Information,This paper discusses the benefits of describ...,cs.IT cs.AI math.IT q-bio.PE,2013-04-05,"[[Gershenson, Carlos, ]]",http://arxiv.org/licenses/nonexclusive-distrib...
670,704.0671,Learning from compressed observations,The problem of statistical learning is to co...,cs.IT cs.LG math.IT,2016-11-15,"[[Raginsky, Maxim, ]]",None
953,704.0954,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun...",cs.IT cs.LG math.IT,2009-11-13,"[[Kar, Soummya, ], [Moura, Jose M. F., ]]",None
...,...,...,...,...,...,...,...
99327,812.3201,Ultrahigh dimensional variable selection: beyo...,Variable selection in high-dimensional space...,stat.ME stat.ML,2008-12-18,"[[Fan, Jianqing, ], [Samworth, Richard, ], [Wu...",http://arxiv.org/licenses/nonexclusive-distrib...
99555,812.3429,Quantum Predictive Learning and Communication ...,We define a new model of quantum learning th...,quant-ph cs.LG,2022-03-29,"[[Gavinsky, Dmytro, ]]",http://arxiv.org/licenses/nonexclusive-distrib...
99591,812.3465,Linearly Parameterized Bandits,We consider bandit problems involving a larg...,cs.LG,2010-02-24,"[[Rusmevichientong, Paat, ], [Tsitsiklis, John...",http://arxiv.org/licenses/nonexclusive-distrib...
99604,812.3478,Automatic Construction of Lightweight Domain O...,The need for domain ontologies in mission cr...,cs.AI,2008-12-19,"[[Wong, Wilson, ], [Liu, Wei, ], [Liaw, Saujoe...",http://arxiv.org/licenses/nonexclusive-distrib...


Retriever agent

Uses:
title + abstract → embeddings
categories → routing/filtering
update_date → ranking

Summarizer agent

Uses:
abstract (primary)
title (context anchor)

Critic agent

Uses:
categories (domain correctness check)
update_date (staleness check)
optional: multiple retrieved abstracts

Synthesis agent

Uses:
top-k summaries + citations via id

In [57]:
cs_rows['license'] = cs_rows['license'].fillna("")

C:\Users\papsr\AppData\Local\Temp\ipykernel_10768\588623107.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cs_rows['license'] = cs_rows['license'].fillna("")


In [58]:
cs_rows

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
46,704.0047,Igor Grabec,T. Kosel and I. Grabec,Intelligent location of simultaneously active ...,"5 pages, 5 eps figures, uses IEEEtran.cls",None,None,None,cs.NE cs.AI,,The intelligent acoustic emission locator is...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2009-09-29,"[[Kosel, T., ], [Grabec, I., ]]"
49,704.0050,Igor Grabec,T. Kosel and I. Grabec,Intelligent location of simultaneously active ...,"5 pages, 7 eps figures, uses IEEEtran.cls",None,None,None,cs.NE cs.AI,,Part I describes an intelligent acoustic emi...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2007-05-23,"[[Kosel, T., ], [Grabec, I., ]]"
303,704.0304,Carlos Gershenson,Carlos Gershenson,The World as Evolving Information,"16 pages. Extended version, three more laws of...","Minai, A., Braha, D., and Bar-Yam, Y., eds. Un...",10.1007/978-3-642-18003-3_10,None,cs.IT cs.AI math.IT q-bio.PE,http://arxiv.org/licenses/nonexclusive-distrib...,This paper discusses the benefits of describ...,"[{'version': 'v1', 'created': 'Tue, 3 Apr 2007...",2013-04-05,"[[Gershenson, Carlos, ]]"
670,704.0671,Maxim Raginsky,Maxim Raginsky,Learning from compressed observations,6 pages; submitted to the 2007 IEEE Informatio...,None,10.1109/ITW.2007.4313111,None,cs.IT cs.LG math.IT,,The problem of statistical learning is to co...,"[{'version': 'v1', 'created': 'Thu, 5 Apr 2007...",2016-11-15,"[[Raginsky, Maxim, ]]"
953,704.0954,Jos\'e M. F. Moura,Soummya Kar and Jose M. F. Moura,Sensor Networks with Random Links: Topology De...,Submitted to IEEE Transactions,None,10.1109/TSP.2008.920143,None,cs.IT cs.LG math.IT,,"In a sensor network, in practice, the commun...","[{'version': 'v1', 'created': 'Fri, 6 Apr 2007...",2009-11-13,"[[Kar, Soummya, ], [Moura, Jose M. F., ]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99327,812.3201,Yichao Wu,"Jianqing Fan, Richard Samworth, Yichao Wu",Ultrahigh dimensional variable selection: beyo...,32 pages,None,None,None,stat.ME stat.ML,http://arxiv.org/licenses/nonexclusive-distrib...,Variable selection in high-dimensional space...,"[{'version': 'v1', 'created': 'Wed, 17 Dec 200...",2008-12-18,"[[Fan, Jianqing, ], [Samworth, Richard, ], [Wu..."
99555,812.3429,Dmytro Gavinsky,Dmytro Gavinsky,Quantum Predictive Learning and Communication ...,None,None,None,None,quant-ph cs.LG,http://arxiv.org/licenses/nonexclusive-distrib...,We define a new model of quantum learning th...,"[{'version': 'v1', 'created': 'Wed, 17 Dec 200...",2022-03-29,"[[Gavinsky, Dmytro, ]]"
99591,812.3465,Paat Rusmevichientong,Paat Rusmevichientong and John N. Tsitsiklis,Linearly Parameterized Bandits,40 pages; updated results and references,None,None,None,cs.LG,http://arxiv.org/licenses/nonexclusive-distrib...,We consider bandit problems involving a larg...,"[{'version': 'v1', 'created': 'Thu, 18 Dec 200...",2010-02-24,"[[Rusmevichientong, Paat, ], [Tsitsiklis, John..."
99604,812.3478,Wilson Wong,"Wilson Wong, Wei Liu, Saujoe Liaw, Nicoletta B...",Automatic Construction of Lightweight Domain O...,In the Proceedings of the 11th Conference on P...,None,None,None,cs.AI,http://arxiv.org/licenses/nonexclusive-distrib...,The need for domain ontologies in mission cr...,"[{'version': 'v1', 'created': 'Thu, 18 Dec 200...",2008-12-19,"[[Wong, Wilson, ], [Liu, Wei, ], [Liaw, Saujoe..."


In [59]:
cs_rows['license'].isna().sum()

np.int64(0)